# NLI Lyrics Corpus Builder (v2)

**Goal:** Build a corpus of English-language songs by artists with a known L1, for Native Language Identification research.

**Scope:** Phases 2–6 of the data collection plan.
- **Phase 2** — Build candidate artist pool (Wikidata + MusicBrainz)
- **Phase 3** — Pre-filter candidates (genre, Wikipedia presence, optional year)
- **Phase 4** — Scrape lyrics from Genius (with error handling)
- **Phase 5** — Language filtering with fastText
- **Phase 6** — Metadata filtering (covers, duplicates)

Manual L1 verification (Phase 7) happens after this notebook on the produced `artists_for_verification.csv`.

**New additions (v2)**
- Tightened genre filter: dropped generic `latin` keyword (was matching bolero/folk artists who don't write in English)
- Added optional year filter to drop pre-1970s artists
- Scrape function wrapped in try/except so one bad artist doesn't kill the whole run
- Plain `tqdm` (not `tqdm.notebook`) to avoid `ipywidgets` dependency issues

## Setup

### Install dependencies

Run once:

```bash
pip install lyricsgenius requests musicbrainzngs pandas fasttext-wheel tqdm python-dotenv
```

### API credentials

You need a **Genius API token** (required for Phase 4). Get one here:
- Go to https://genius.com/api-clients
- Sign up / log in, click "New API Client"
- Fill in any app name and website (e.g., `http://example.com`)
- Copy the **Client Access Token**

Create a `.env` file in the same folder as this notebook:

```
GENIUS_TOKEN=your_token_here
```

**Make sure `.env` is in your `.gitignore`** before committing anything.

In [1]:
import os
import json
import time
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import requests
from tqdm import tqdm  # Plain tqdm (no ipywidgets dependency)
from dotenv import load_dotenv

load_dotenv()

GENIUS_TOKEN = os.getenv("GENIUS_TOKEN")
assert GENIUS_TOKEN, "Set GENIUS_TOKEN in .env before running Phase 4"

print("Genius token loaded")

Genius token loaded


/Users/nataliajimenez/miniforge3/envs/auc/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Configuration — change this block for each L1

Everything downstream reads from `CONFIG`. To swap languages, edit this cell and re-run.

In [29]:
CONFIG = {
    # L1 label used in output files
    "L1": "french",

    # ISO 3166-1 alpha-2 country codes where this L1 is dominant
    # France, Monaco. Belgium and Switzerland excluded — French is one of
    # several official languages there, so country alone is not a reliable
    # L1 proxy. Quebec (CA) excluded for the same reason: anglophone Canadians
    # would be misclassified as French-L1.
    "countries": [
        "FR", "MC", "BE"
    ],

    # fastText language code for this L1 (for detecting/excluding it in lyrics)
    "l1_fasttext_code": "fr",

    # Wikidata occupation QIDs to include
    # rapper, hip-hop musician, singer, singer-songwriter
    "occupation_qids": ["Q2252262", "Q177220", "Q753110", "Q488205"],

    # Genre keywords for Phase 3 filter
    # chanson / variété française deliberately excluded — those artists almost
    # exclusively write in French. Focus on genres where French artists have
    # crossed over to English.
    "target_genres": [
        "hip hop", "hip-hop", "rap", "trap", "drill",
        "r&b", "rnb", "pop", "dance", "electronic", "house",
        "indie", "rock", "urban",
    ],

    # Optional: exclude artists born before this year (None to disable)
    # 1975 cuts most pre-modern-pop French artists who wrote exclusively in French
    "min_birth_year": 1970,

    # Phase 5 thresholds
    "min_english_ratio": 0.5,
    "borderline_ratio": 0.3,
    "min_lines_per_song": 8,

    # Phase 8 requirement
    "min_songs_per_artist": 1,

    # Output structure
    "data_dir": Path("./data"),
}

DATA = CONFIG["data_dir"] / CONFIG["L1"]
for sub in ["raw", "interim", "processed", "logs"]:
    (DATA / sub).mkdir(parents=True, exist_ok=True)

print(f"Workspace: {DATA.resolve()}")
print(f"L1: {CONFIG['L1']} | Countries: {len(CONFIG['countries'])} | fastText code: {CONFIG['l1_fasttext_code']}")


Workspace: /Users/nataliajimenez/Desktop/AUC/TM/project/data/french
L1: french | Countries: 3 | fastText code: fr


---
# Phase 2 — Candidate artist pool

## 2A — Wikidata SPARQL

In [30]:
def iso_to_wikidata_country(iso_codes):
    """Map ISO country codes to Wikidata QIDs."""
    mapping = {
        "ES": "Q29",  "MX": "Q96",  "AR": "Q414", "CO": "Q739", "PE": "Q419",
        "CL": "Q298", "VE": "Q717", "EC": "Q736", "GT": "Q774", "CU": "Q241",
        "DO": "Q786", "HN": "Q783", "BO": "Q750", "SV": "Q792", "PY": "Q733",
        "NI": "Q811", "CR": "Q800", "PR": "Q1183","PA": "Q804", "UY": "Q77",
        "IT": "Q38",  "SM": "Q238", "VA": "Q237", "NL": "Q55",  "BE": "Q31",
        "RU": "Q159", "BY": "Q184", "UA": "Q212", "KZ": "Q232", "US": "Q30",
        "GB": "Q145", "CA": "Q16",  "AU": "Q408", "IE": "Q27",  "NZ": "Q664",
        "FR": "Q142", "MC": "Q235",
    }
    return [mapping[c] for c in iso_codes if c in mapping]


def build_wikidata_query(country_qids, occupation_qids):
    countries_block = " ".join(f"wd:{q}" for q in country_qids)
    occupations_block = " ".join(f"wd:{q}" for q in occupation_qids)
    return f"""
    SELECT DISTINCT ?artist ?artistLabel ?birthPlaceLabel ?birthYear ?genreLabel ?enwiki ?genius
    WHERE {{
      ?artist wdt:P31 wd:Q5 ;
              wdt:P106 ?occupation ;
              wdt:P27 ?country .
      VALUES ?occupation {{ {occupations_block} }}
      VALUES ?country {{ {countries_block} }}
      OPTIONAL {{ ?artist wdt:P19 ?birthPlace . }}
      OPTIONAL {{ ?artist wdt:P569 ?birthDate . BIND(YEAR(?birthDate) AS ?birthYear) }}
      OPTIONAL {{ ?artist wdt:P136 ?genre . }}
      OPTIONAL {{ ?artist wdt:P2373 ?genius . }}
      OPTIONAL {{
        ?enwiki schema:about ?artist ;
                schema:isPartOf <https://en.wikipedia.org/> .
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
    }}
    LIMIT 5000
    """


def query_wikidata(sparql_query):
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "NLI-research-notebook/1.0 (academic, AUC text mining course)",
    }
    response = requests.get(url, params={"query": sparql_query}, headers=headers, timeout=120)
    response.raise_for_status()
    return response.json()


# Per-(country, occupation) loop
country_qids = iso_to_wikidata_country(CONFIG["countries"])
occupation_qids = CONFIG["occupation_qids"]
print(f"Querying Wikidata for {len(country_qids)} countries × {len(occupation_qids)} occupations (one at a time)...")

all_bindings = []
failed = []

for country_qid in country_qids:
    for occ_qid in occupation_qids:
        query = build_wikidata_query([country_qid], [occ_qid])
        print(f"  {country_qid} × {occ_qid}...", end=" ", flush=True)
        success = False
        for attempt in range(3):
            try:
                raw = query_wikidata(query)
                chunk = raw["results"]["bindings"]
                print(f"{len(chunk)} rows")
                all_bindings.extend(chunk)
                success = True
                break
            except (requests.RequestException, ValueError) as e:
                print(f"[attempt {attempt+1}: {type(e).__name__}]", end=" ", flush=True)
                time.sleep(5 * (attempt + 1))
        if not success:
            print(f"GAVE UP on {country_qid} × {occ_qid}")
            failed.append((country_qid, occ_qid))
        time.sleep(1)

bindings = all_bindings
print(f"\nTotal: {len(bindings)} rows; {len(failed)} (country, occupation) pairs failed")
if failed:
    print(f"⚠️  Failed: {failed}")

Querying Wikidata for 3 countries × 4 occupations (one at a time)...
  Q142 × Q2252262... 855 rows
  Q142 × Q177220... 4778 rows
  Q142 × Q753110... [attempt 1: JSONDecodeError] [attempt 2: JSONDecodeError] [attempt 3: JSONDecodeError] GAVE UP on Q142 × Q753110
  Q142 × Q488205... 1854 rows
  Q235 × Q2252262... 0 rows
  Q235 × Q177220... 8 rows
  Q235 × Q753110... 5 rows
  Q235 × Q488205... 5 rows
  Q31 × Q2252262... 126 rows
  Q31 × Q177220... 1321 rows
  Q31 × Q753110... 109 rows
  Q31 × Q488205... 195 rows

Total: 9256 rows; 1 (country, occupation) pairs failed
⚠️  Failed: [('Q142', 'Q753110')]


In [31]:
def parse_wikidata_results(bindings):
    rows = []
    for b in bindings:
        rows.append({
            "wikidata_id": b["artist"]["value"].rsplit("/", 1)[-1],
            "name": b.get("artistLabel", {}).get("value", ""),
            "birthplace": b.get("birthPlaceLabel", {}).get("value", ""),
            "birth_year": b.get("birthYear", {}).get("value", ""),
            "genre": b.get("genreLabel", {}).get("value", ""),
            "enwiki_url": b.get("enwiki", {}).get("value", ""),
            "genius_slug": b.get("genius", {}).get("value", ""),
        })
    df = pd.DataFrame(rows)
    # Collapse multi-genre duplicates: keep one row per artist, join genres
    df = (
        df.groupby(["wikidata_id", "name", "birthplace", "birth_year", "enwiki_url", "genius_slug"],
                   dropna=False, as_index=False)
          .agg({"genre": lambda s: "; ".join(sorted({g for g in s if g}))})
    )
    df["source"] = "wikidata"
    return df


candidates_wd = parse_wikidata_results(bindings)
candidates_wd.to_csv(DATA / "interim" / "candidates_wikidata.csv", index=False)
print(f"Wikidata: {len(candidates_wd)} unique artists")
candidates_wd.head(10)

Wikidata: 6496 unique artists


,wikidata_id,name,birthplace,birth_year,enwiki_url,genius_slug,genre,source
0,Q100144813,Colette Herzog,Strasbourg,1923,,,,wikidata
1,Q100251627,P.R2B,Saint-Doulchard,1990,https://en.wikipedia.org/wiki/P.R2B,,,wikidata
2,Q100276051,Remy Ray,Machelen,1932,,,,wikidata
3,Q100316237,Lili Cros,,,,,,wikidata
4,Q100316238,Thierry Chazelle,,,,,,wikidata
5,Q100361853,Pablo Grant,Berlin,1997,https://en.wikipedia.org/wiki/Pablo_Grant,,,wikidata
6,Q100381305,Huero,,1997,,,hip-hop,wikidata
7,Q100533408,Lady Lova,,,https://en.wikipedia.org/wiki/Lady_Lova,,,wikidata
8,Q100537726,Aelis Loddo,,,,,Occitan folk music,wikidata
9,Q100587672,Fatou Samba,Dakar,1995,https://en.wikipedia.org/wiki/Fatou_Samba,,K-pop,wikidata


## 2B — MusicBrainz

In [32]:
import musicbrainzngs

musicbrainzngs.set_useragent("NLI-research-notebook", "1.0", "academic@auc.nl")


def fetch_musicbrainz_artists(country_codes, tags=("hip hop", "rap", "reggaeton"), limit=100):
    rows = []
    for cc in tqdm(country_codes, desc="MusicBrainz countries"):
        for tag in tags:
            query = f'country:{cc} AND tag:"{tag}"'
            try:
                result = musicbrainzngs.search_artists(query=query, limit=limit)
            except musicbrainzngs.WebServiceError as e:
                print(f"  {cc}/{tag}: error {e}")
                time.sleep(2)
                continue
            for a in result.get("artist-list", []):
                rows.append({
                    "musicbrainz_id": a.get("id"),
                    "name": a.get("name"),
                    "country": a.get("country"),
                    "tags": "; ".join(t["name"] for t in a.get("tag-list", [])) if a.get("tag-list") else "",
                    "disambiguation": a.get("disambiguation", ""),
                    "source": "musicbrainz",
                })
            time.sleep(1.0)  # MusicBrainz requires 1 req/sec
    return pd.DataFrame(rows).drop_duplicates(subset="musicbrainz_id")


candidates_mb = fetch_musicbrainz_artists(CONFIG["countries"])
candidates_mb.to_csv(DATA / "interim" / "candidates_musicbrainz.csv", index=False)
print(f"MusicBrainz: {len(candidates_mb)} unique artists")
candidates_mb.head(10)

MusicBrainz countries: 100%|██████████████████████████████████████████████████████████████| 3/3 [00:10<00:00,  3.59s/it]

MusicBrainz: 287 unique artists


,musicbrainz_id,name,country,tags,disambiguation,source
0,7aa4a16e-9d5a-41d3-968e-fae1a2c84d05,IAM,FR,marseille; hip hop; french rap; french,French rap band,musicbrainz
1,8e6c44df-06fc-4852-83d0-ef0efed6630a,Jul,FR,hip hop; pop rap,French rapper,musicbrainz
2,b2fbd053-4380-412c-95d2-35c6da8f1011,GIMS,FR,pop; pop rap; hardcore hip hop; french hip hop...,"French-Congolese rapper, Maître Gims",musicbrainz
3,f4a92aac-995b-47ab-80a5-33218a3e7823,Rohff,FR,hip hop,French rapper,musicbrainz
4,0135e5ec-4ffd-46e5-9c3c-566b3df46a8d,Suprême NTM,FR,hip hop,,musicbrainz
5,d2e123d0-c53e-4444-a52a-efeff44953c7,DJ Cam,FR,electronic; downtempo; instrumental; hip hop; ...,French jazzy hip-hop DJ/producer,musicbrainz
6,998f1ece-a2b0-47d2-93df-17553430ab73,La Fouine,FR,hip hop,French rapper,musicbrainz
7,062ab94e-b7a5-47a9-b3f4-cc1750c3f859,Vald,FR,hip hop,French rapper,musicbrainz
8,bbbd2644-b4cb-4bb5-a442-315310f68a0b,MC Solaar,FR,french; hip hop; hiphop; pop rap; european; hi...,,musicbrainz
9,3b1263a9-da8c-4c14-8877-15f5b47ca34e,LIM,FR,hip hop; hardcore hip hop,French hip-hop artist,musicbrainz


## 2C — Merge candidates from both sources

In [33]:
def normalize_name(s):
    if pd.isna(s):
        return ""
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


wd = candidates_wd.copy()
mb = candidates_mb.copy()
wd["name_norm"] = wd["name"].apply(normalize_name)
mb["name_norm"] = mb["name"].apply(normalize_name)

merged = wd.merge(mb, on="name_norm", how="outer", suffixes=("_wd", "_mb"))
merged["name"] = merged["name_wd"].fillna(merged["name_mb"])
merged["sources"] = merged.apply(
    lambda r: "; ".join([s for s in [r.get("source_wd"), r.get("source_mb")] if pd.notna(s)]),
    axis=1,
)

candidates = merged[["name", "name_norm", "sources", "wikidata_id", "musicbrainz_id",
                     "birthplace", "birth_year", "genre", "country", "tags",
                     "enwiki_url", "genius_slug", "disambiguation"]].drop_duplicates("name_norm")

candidates.to_csv(DATA / "interim" / "candidates_merged.csv", index=False)
print(f"Merged candidates: {len(candidates)}")
print(f"  Wikidata only:    {(merged['source_wd'].notna() & merged['source_mb'].isna()).sum()}")
print(f"  MusicBrainz only: {(merged['source_mb'].notna() & merged['source_wd'].isna()).sum()}")
print(f"  Both sources:     {(merged['source_wd'].notna() & merged['source_mb'].notna()).sum()}")

Merged candidates: 6568
  Wikidata only:    6376
  MusicBrainz only: 171
  Both sources:     120


---
# Phase 3 — Pre-filter

Aggressive filtering before scraping. Each filter is named so you can see what dropped what.

**Why these filters:**
1. **Genre filter** — only keep hip-hop/rap/urban genres where English-language tracks are common. Generic "latin" tag deliberately excluded.
2. **Wikipedia filter** — require an English Wikipedia page. This is a proxy for "documented artist with verifiable bio", which we'll need anyway for L1 verification (Phase 7).
3. **Year filter (optional)** — drop pre-1970 artists who are unlikely to have English-language modern hip-hop releases.

In [34]:
# load data if not first time running
candidates = pd.read_csv(DATA / "interim" / "candidates_merged.csv")
print(f"Loaded {len(candidates)} existing candidates from disk")

Loaded 6568 existing candidates from disk


In [35]:
pre_filtered = candidates.copy()
print(f"Starting candidates:        {len(pre_filtered)}")

# Filter 1: name not empty
pre_filtered = pre_filtered[pre_filtered["name"].str.len() > 0]
print(f"After name filter:          {len(pre_filtered)}")

# Filter 2: relevant genre (hip-hop/rap/urban, not generic 'latin')
def has_target_genre(row):
    genres = " ".join(str(x).lower() for x in [row.get("genre", ""), row.get("tags", "")])
    return any(kw in genres for kw in CONFIG["target_genres"])

pre_filtered = pre_filtered[pre_filtered.apply(has_target_genre, axis=1)]
print(f"After genre filter:         {len(pre_filtered)}")

# Filter 3: must have an English Wikipedia page
pre_filtered = pre_filtered[
    pre_filtered["enwiki_url"].notna() & (pre_filtered["enwiki_url"] != "")
]
print(f"After Wikipedia filter:     {len(pre_filtered)}")

# Filter 4 (optional): birth year cutoff
if CONFIG["min_birth_year"] is not None:
    # Convert birth_year to numeric (some are strings, some empty)
    pre_filtered["birth_year_num"] = pd.to_numeric(pre_filtered["birth_year"], errors="coerce")
    # Keep rows where birth_year is unknown (NaN) OR >= cutoff
    mask = pre_filtered["birth_year_num"].isna() | (pre_filtered["birth_year_num"] >= CONFIG["min_birth_year"])
    pre_filtered = pre_filtered[mask]
    print(f"After year filter (>={CONFIG['min_birth_year']}):  {len(pre_filtered)}")

pre_filtered = pre_filtered[pre_filtered["country"].notna()]
print(f"After country filter:       {len(pre_filtered)}")

pre_filtered.to_csv(DATA / "interim" / "candidates_prefiltered.csv", index=False)

n = len(pre_filtered)
est_min = n * 30 / 60
print(f"\nEstimated scrape time: {est_min:.0f} min ({est_min/60:.1f} hours) at max_songs=25")

if n < 100:
    print("Good, proceed with scraping")
elif n < 300:
    print("Acceptable, proceed but be patient")
else:
    print(f"Still too many ({n}). Consider tightening CONFIG['target_genres'] or raising min_birth_year.")

print("\nSample of survivors (do these names look like artists who would write English songs?):")
pre_filtered[["name", "country", "birth_year", "genre", "tags"]].head(20)

Starting candidates:        6568
After name filter:          6568
After genre filter:         1436
After Wikipedia filter:     741
After year filter (>=1970):  481
After country filter:       84

Estimated scrape time: 42 min (0.7 hours) at max_songs=25
Good, proceed with scraping

Sample of survivors (do these names look like artists who would write English songs?):


,name,country,birth_year,genre,tags
166,Alibi Montana,FR,1978.0,hip-hop,hip hop; gangsta rap; underground hip hop
191,Alizée,FR,1984.0,dance-pop; electropop; pop music,rock; pop; french; dance; france; francophone;...
201,Alonzo,FR,1982.0,gangsta rap; hip-hop,hip hop; pop rap; trap
202,Aloïse Sauvage,FR,1992.0,hip-hop,pop rap
205,Alpha Wann,FR,1989.0,French hip-hop,rap; hip hop; french rap; conscious hip hop; e...
459,Ateyaba,FR,1989.0,hip-hop; rapping,rap; electro; france; hip hop; alternative hip...
528,Baloji,BE,1978.0,hip-hop,rap; hip hop; funk; world; afrobeat; world bea...
637,Berry,BE,1978.0,pop music,hip hop
678,Black M,FR,1984.0,contemporary R&B; hip-hop,hip hop
680,Blacko,FR,1979.0,hip-hop,raga; rap; reggae


In [36]:
pre_filtered[["name", "country", "birth_year", "genre"]].to_string()

"                     name country  birth_year                                                                                                                    genre\n166         Alibi Montana      FR      1978.0                                                                                                                  hip-hop\n191                Alizée      FR      1984.0                                                                                         dance-pop; electropop; pop music\n201                Alonzo      FR      1982.0                                                                                                     gangsta rap; hip-hop\n202        Aloïse Sauvage      FR      1992.0                                                                                                                  hip-hop\n205            Alpha Wann      FR      1989.0                                                                                                           French 

By looking at this output I have decided to remove artists which may add noise by having ghostwriters and artists that are not native french speakers (remove/change for other languages)

In [37]:
# Drop high ghostwriter risk before scraping
# French high-ghostwriter-risk / non-L1 artists.
# Conservative list — major-label pop artists with known co-writing teams,
# Eurovision/international-pop crossover artists, and foreign-born artists
# that Wikidata may have flagged as French (citizenship ≠ L1).
ghostwriter_risk = [
    # Major-label pop with extensive co-writer credits / English-language teams
    "David Guetta", "Bob Sinclar", "Martin Solveig",
    "Christine and the Queens", "Indila", "Zaz",
    # Eurovision / international-pop crossover
    "Amir", "Bilal Hassani", "Alma",
    # Anglophone Canadians sometimes tagged as French via Quebec heritage / dual citizenship:
    "Avril Lavigne", "Céline Dion", "Shania Twain",
    # Other foreign-born or anglophone-raised:
    "Alireza JJ",  # Iranian-French, L1 likely Persian
    "Ana Tijoux",  # Chilean-French, L1 Spanish
    # Add as needed during manual verification
]

before = len(pre_filtered)
pre_filtered = pre_filtered[~pre_filtered["name"].isin(ghostwriter_risk)]
print(f"Dropped {before - len(pre_filtered)} ghostwriter-risk / non-L1 artists")
print(f"Remaining: {len(pre_filtered)}")

Dropped 0 ghostwriter-risk / non-L1 artists
Remaining: 84


---
# Phase 4 — Scrape lyrics from Genius

**Checkpointed:** each artist's scrape saves to `data/<L1>/raw/<artist_slug>.json`. If the loop crashes or you interrupt it, re-running skips artists already done.

**Wrapped in try/except:** one bad artist won't kill the whole run.

Firts we perform a pre-scrapping sanity check:

In [38]:
print(f"About to scrape {len(pre_filtered)} artists")
print(f"Estimated time: {len(pre_filtered) * 30 / 60:.0f} minutes")
print("\nFirst 10:")
print(pre_filtered[["name", "country"]].head(10).to_string())
print("\nLast 10:")
print(pre_filtered[["name", "country"]].tail(10).to_string())

About to scrape 84 artists
Estimated time: 42 minutes

First 10:
               name country
166   Alibi Montana      FR
191          Alizée      FR
201          Alonzo      FR
202  Aloïse Sauvage      FR
205      Alpha Wann      FR
459         Ateyaba      FR
528          Baloji      BE
637           Berry      BE
678         Black M      FR
680          Blacko      FR

Last 10:
                 name country
5914           Sultan      FR
6029         Theodora      FR
6077          Tiakola      FR
6190             Vald      FR
6223        Vegedream      FR
6316          Werenoi      FR
6389          Yannick      FR
6423  Youssef Swatt's      BE
6424       Youssoupha      FR
6505             Zola      FR


In [39]:
import lyricsgenius

genius = lyricsgenius.Genius(
    GENIUS_TOKEN,
    remove_section_headers=True,
    skip_non_songs=True,
    excluded_terms=["(Remix)", "(Live)", "(Acoustic)", "(Demo)", "(Instrumental)"],
    timeout=30,
    retries=3,
)
genius.verbose = False  # Set as attribute, not constructor arg (compatibility fix)
print("Genius client initialized")

Genius client initialized


In [2]:
def slugify(name):
    s = re.sub(r"[^\w\s-]", "", str(name).lower()).strip()
    return re.sub(r"[-\s]+", "_", s)


def scrape_artist(name, max_songs=25):
    """Scrape discography; return dict with songs or error.
    Compatible with lyricsgenius 3.12+ which uses to_dict() for IDs.
    """
    try:
        artist = genius.search_artist(name, max_songs=max_songs, sort="popularity")
    except Exception as e:
        return {"error": str(e), "name": name}
    if artist is None:
        return {"error": "not_found", "name": name}
    
    # Get artist ID from to_dict
    try:
        artist_id = artist.to_dict().get("id")
    except Exception:
        artist_id = None
    
    songs = []
    for song in artist.songs:
        try:
            song_dict = song.to_dict()
            song_id = song_dict.get("id")
        except Exception:
            song_id = None
        
        # featured_artists handling - check if it's a list of dicts or objects
        featured = []
        try:
            for fa in (song.featured_artists or []):
                if isinstance(fa, dict):
                    featured.append(fa.get("name", ""))
                elif hasattr(fa, "name"):
                    featured.append(fa.name)
        except Exception:
            pass
        
        songs.append({
            "song_id": song_id,
            "title": song.title,
            "lyrics": song.lyrics or "",
            "url": song.url,
            "featured_artists": featured,
        })
    
    return {"name": name, "genius_id": artist_id, "songs": songs}


def scrape_all(candidates_df, max_songs=25):
    """Loop over candidates with checkpointing and error handling."""
    raw_dir = DATA / "raw"
    log_path = DATA / "logs" / "scrape_log.jsonl"

    for _, row in tqdm(candidates_df.iterrows(), total=len(candidates_df), desc="Scraping Genius"):
        slug = slugify(row["name"])
        if not slug:
            continue
        out_path = raw_dir / f"{slug}.json"
        if out_path.exists():
            continue  # Checkpoint: skip already scraped

        # Wrap entire scrape attempt so one bad artist doesn't kill the run
        try:
            result = scrape_artist(row["name"], max_songs=max_songs)
        except Exception as e:
            result = {"error": f"unexpected: {e}", "name": row["name"]}

        try:
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            with open(log_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "name": row["name"],
                    "slug": slug,
                    "status": "error" if "error" in result else "ok",
                    "n_songs": len(result.get("songs", [])),
                }) + "\n")
        except Exception as e:
            print(f"Failed to save {slug}: {e}")

        time.sleep(1.0)  # Be polite to Genius

print("Scrape function ready.")

Scrape function ready.


In [41]:
# Kick off the scrape. This can take a while - check the time estimate above.
# If you interrupt, re-running this cell resumes from where it stopped (checkpointed).
scrape_all(pre_filtered, max_songs=25)
print("Scrape complete")

Scraping Genius: 100%|██████████████████████████████████████████████████████████████████| 84/84 [35:44<00:00, 25.53s/it]

Scrape complete


In [42]:
# Load all scraped data into a song-level DataFrame
def load_scraped_songs():
    rows = []
    n_errors = 0
    for json_path in (DATA / "raw").glob("*.json"):
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        if "error" in data:
            n_errors += 1
            continue
        for song in data.get("songs", []):
            rows.append({
                "artist_name": data["name"],
                "artist_slug": json_path.stem,
                "genius_artist_id": data.get("genius_id"),
                "song_id": song["song_id"],
                "title": song["title"],
                "lyrics": song["lyrics"],
                "url": song["url"],
                "featured_artists": "; ".join(song["featured_artists"]),
            })
    print(f"Loaded {len(rows)} songs from successful scrapes ({n_errors} artists had errors)")
    return pd.DataFrame(rows)


songs_raw = load_scraped_songs()
print(f"Unique artists with songs: {songs_raw['artist_slug'].nunique()}")
songs_raw.head()

Loaded 2111 songs from successful scrapes (0 artists had errors)
Unique artists with songs: 85


,artist_name,artist_slug,genius_artist_id,song_id,title,lyrics,url,featured_artists
0,Sultan,sultan,6670,71898,4 Étoiles,Ewa Moinama\nHisse le drapeau comorien\nCroiss...,https://genius.com/Sultan-4-etoiles-lyrics,Rohff
1,Sultan,sultan,6670,96284,Mec à meuf,"Depuis l'école primaire, j'ai scruté tous les ...",https://genius.com/Sultan-mec-a-meuf-lyrics,
2,Sultan,sultan,6670,468815,Que d’la peufra,Wesh ... (Que D'la peu-fra)\nTookie (Que D'la ...,https://genius.com/Sultan-que-dla-peufra-lyrics,Ixzo
3,Sultan,sultan,6670,92852,Sois fier de c’que t’es,Sois fier de... tes valeurs avant celles de l'...,https://genius.com/Sultan-sois-fier-de-cque-te...,
4,Sultan,sultan,6670,404953,Chez Wam,Okok\nCondamné à régner bande d'enfoiré\nBo So...,https://genius.com/Sultan-chez-wam-lyrics,Croma; P.O.P. (Rapper)


---
# Phase 5 — Language filtering with fastText

In [43]:
import urllib.request

FASTTEXT_MODEL_PATH = DATA / "lid.176.ftz"
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz"

if not FASTTEXT_MODEL_PATH.exists():
    print("Downloading fastText language model (~1MB)...")
    urllib.request.urlretrieve(FASTTEXT_URL, FASTTEXT_MODEL_PATH)
    print(f"Saved to {FASTTEXT_MODEL_PATH}")
else:
    print(f"Model already present at {FASTTEXT_MODEL_PATH}")

import fasttext
fasttext.FastText.eprint = lambda *a, **k: None  # Suppress deprecation warning
ft_model = fasttext.load_model(str(FASTTEXT_MODEL_PATH))
print("fastText model loaded")

Model already present at data/french/lid.176.ftz
fastText model loaded


In [44]:
def clean_lyrics(text):
    """Light cleaning before language ID."""
    if not text:
        return ""
    text = re.sub(r"\[[^\]]+\]", "", text)  # Strip [Verse], [Chorus] etc.
    text = re.sub(r"^\d+\s+Contributors.*?Lyrics", "", text, flags=re.DOTALL)  # Genius header
    text = re.sub(r"\d*Embed\s*$", "", text)  # Genius footer
    text = re.sub(r"\s+", " ", text).strip()
    return text


def analyze_song_language(text, ft):
    """Per-song language stats based on line-level classification."""
    cleaned = clean_lyrics(text)
    lines = [ln.strip() for ln in re.split(r"[\n.!?]", cleaned) if ln.strip() and len(ln.strip()) > 3]
    if not lines:
        return {"n_lines": 0, "english_ratio": 0, "l1_ratio": 0, "top_lang": None}
    labels, _ = ft.predict(lines, k=1)
    lang_codes = [lbl[0].replace("__label__", "") for lbl in labels]
    lang_counts = Counter(lang_codes)
    total = len(lang_codes)
    return {
        "n_lines": total,
        "english_ratio": lang_counts.get("en", 0) / total,
        "l1_ratio": lang_counts.get(CONFIG["l1_fasttext_code"], 0) / total,
        "top_lang": lang_counts.most_common(1)[0][0],
    }


lang_stats = []
for _, row in tqdm(songs_raw.iterrows(), total=len(songs_raw), desc="Language ID"):
    lang_stats.append(analyze_song_language(row["lyrics"], ft_model))

lang_df = pd.DataFrame(lang_stats)
songs_with_lang = pd.concat([songs_raw.reset_index(drop=True), lang_df], axis=1)
songs_with_lang["clean_lyrics"] = songs_with_lang["lyrics"].apply(clean_lyrics)
songs_with_lang.to_csv(DATA / "interim" / "songs_with_language.csv", index=False)

print(f"\nSongs analyzed: {len(songs_with_lang)}")
print("\nTop language distribution:")
print(songs_with_lang["top_lang"].value_counts().head(10))

Language ID: 100%|████████████████████████████████████████████████████████████████| 2111/2111 [00:00<00:00, 4212.88it/s]



Songs analyzed: 2111

Top language distribution:
top_lang
fr    1987
en      93
nl      10
es       7
de       3
pt       3
sw       2
hu       1
tl       1
kw       1
Name: count, dtype: int64


In [45]:
# Apply thresholds
qualifying = songs_with_lang[
    (songs_with_lang["english_ratio"] >= CONFIG["min_english_ratio"]) &
    (songs_with_lang["n_lines"] >= CONFIG["min_lines_per_song"])
].copy()

borderline = songs_with_lang[
    (songs_with_lang["english_ratio"] >= CONFIG["borderline_ratio"]) &
    (songs_with_lang["english_ratio"] < CONFIG["min_english_ratio"]) &
    (songs_with_lang["n_lines"] >= CONFIG["min_lines_per_song"])
].copy()

qualifying.to_csv(DATA / "interim" / "songs_english.csv", index=False)
borderline.to_csv(DATA / "interim" / "songs_borderline_review.csv", index=False)

print(f"Qualifying (english_ratio >= {CONFIG['min_english_ratio']}): {len(qualifying)} songs by {qualifying['artist_slug'].nunique()} artists")
print(f"Borderline (manual review): {len(borderline)} songs")

Qualifying (english_ratio >= 0.5): 12 songs by 10 artists
Borderline (manual review): 15 songs


### Line extraction songs

In [46]:
def extract_english_lines(raw_text, ft, min_confidence=0.7, min_line_length=5):
    """Return only the lines classified as English from a song.
    
    Args:
        raw_text: Original lyrics with newlines preserved
        ft: loaded fastText model
        min_confidence: Minimum probability for English classification (0-1)
        min_line_length: Skip lines shorter than this many characters
    
    Returns:
        dict with: english_only_text, n_english_lines, n_total_lines, english_ratio
    """
    if not raw_text or not isinstance(raw_text, str):
        return {
            "english_only_text": "",
            "n_english_lines": 0,
            "n_total_lines": 0,
            "english_ratio": 0,
        }
    
    # Strip section headers and Genius boilerplate, keep newlines
    text = re.sub(r"\[[^\]]+\]", "", raw_text)
    text = re.sub(r"^\d+\s+Contributors.*?Lyrics", "", text, flags=re.DOTALL)
    text = re.sub(r"\d*Embed\s*$", "", text)
    
    # Split on actual newlines
    lines = [ln.strip() for ln in text.split("\n") if ln.strip() and len(ln.strip()) >= min_line_length]
    
    if not lines:
        return {
            "english_only_text": "",
            "n_english_lines": 0,
            "n_total_lines": 0,
            "english_ratio": 0,
        }
    
    # Predict per line
    labels, probs = ft.predict(lines, k=1)
    
    english_lines = []
    for line, lbl, prob in zip(lines, labels, probs):
        is_english = lbl[0] == "__label__en"
        confident = prob[0] >= min_confidence
        if is_english and confident:
            english_lines.append(line)
    
    return {
        "english_only_text": "\n".join(english_lines),
        "n_english_lines": len(english_lines),
        "n_total_lines": len(lines),
        "english_ratio": len(english_lines) / len(lines) if lines else 0,
    }


# Apply to all scraped songs (use songs_raw with original lyrics, not cleaned)
print("Extracting English-only lines from each song...")
extracted = []
for _, row in tqdm(songs_raw.iterrows(), total=len(songs_raw)):
    extracted.append(extract_english_lines(row["lyrics"], ft_model))

extract_df = pd.DataFrame(extracted)

# Combine with song metadata
songs_english_only = pd.concat([songs_raw.reset_index(drop=True), extract_df], axis=1)

# Add word count of extracted English text
songs_english_only["english_word_count"] = songs_english_only["english_only_text"].str.split().str.len()

# Save full results before filtering (for debugging)
songs_english_only.to_csv(DATA / "interim" / "songs_english_extracted.csv", index=False)

# Filter to songs with meaningful English content
MIN_ENGLISH_WORDS = 100  # 100 words is roughly a verse

usable = songs_english_only[
    songs_english_only["english_word_count"] >= MIN_ENGLISH_WORDS
].copy()

print(f"\n{'='*60}")
print(f"RESULTS")
print(f"{'='*60}")
print(f"Total scraped songs: {len(songs_english_only)}")
print(f"Songs with >={MIN_ENGLISH_WORDS} English words: {len(usable)}")
print(f"Unique artists: {usable['artist_slug'].nunique()}")
print(f"\nDistribution of English word counts:")
print(songs_english_only["english_word_count"].describe())

print(f"\nTop 20 songs by English word count:")
print(usable.sort_values("english_word_count", ascending=False)[
    ["artist_name", "title", "n_english_lines", "n_total_lines", "english_word_count"]
].head(20).to_string())

# Save the usable corpus
usable.to_csv(DATA / "processed" / "english_extracted_corpus_french.csv", index=False)
print(f"\nSaved to: {DATA / 'processed' / 'english_extracted_corpus_italian.csv'}")

Extracting English-only lines from each song...


100%|█████████████████████████████████████████████████████████████████████████████| 2111/2111 [00:00<00:00, 4101.76it/s]



RESULTS
Total scraped songs: 2111
Songs with >=100 English words: 80
Unique artists: 21

Distribution of English word counts:
count    2111.000000
mean       11.057793
std        50.153485
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       610.000000
Name: english_word_count, dtype: float64

Top 20 songs by English word count:
     artist_name                    title  n_english_lines  n_total_lines  english_word_count
979    M. Pokora  They Talk Sh#t About Me              108            121                 610
1499     Ménélik     Failure Meet Success               59             62                 532
2081      Kaaris                  Crystal               37             74                 525
1509     Ménélik                Smoke One               57             65                 509
1491     Ménélik          Relax Your Mind               49             71                 457
975    M. Pokora                Dangerous               51     

---
# Phase 6 — Metadata filtering

In [47]:
COVER_PATTERNS = [
    r"\(cover\)", r"\[cover\]", r"- cover$",
    r"\(remix\)", r"\[remix\]",
    r"\(live\)", r"\[live\]",
    r"\(acoustic\)", r"\(demo\)", r"\(instrumental\)",
]


def is_likely_cover_or_variant(title):
    if not isinstance(title, str):
        return False
    t = title.lower()
    return any(re.search(p, t) for p in COVER_PATTERNS)


def lyrics_fingerprint(text):
    if not isinstance(text, str):
        return ""
    t = re.sub(r"[^\w\s]", "", text.lower())
    t = re.sub(r"\s+", " ", t).strip()
    return t[:500]


# Use the line-extracted corpus (usable), not the legacy song-level filter (qualifying)
filtered = usable.copy()
filtered["has_features"] = filtered["featured_artists"].str.len() > 0
filtered["is_cover_variant"] = filtered["title"].apply(is_likely_cover_or_variant)
# Fingerprint on the extracted English text, not the original French-heavy lyrics
filtered["fingerprint"] = filtered["english_only_text"].apply(lyrics_fingerprint)

before = len(filtered)
filtered = filtered[~filtered["is_cover_variant"]]
print(f"Dropped covers/variants: {before - len(filtered)} → {len(filtered)} remaining")

before = len(filtered)
filtered = filtered.drop_duplicates(subset=["artist_slug", "fingerprint"])
print(f"Dropped lyrics duplicates: {before - len(filtered)} → {len(filtered)} remaining")

print(f"\nSongs with features (flagged, not dropped): {filtered['has_features'].sum()}")

filtered.to_csv(DATA / "interim" / "songs_filtered.csv", index=False)

Dropped covers/variants: 0 → 80 remaining
Dropped lyrics duplicates: 0 → 80 remaining

Songs with features (flagged, not dropped): 27


---
# Handoff to Phase 7 (manual L1 verification)

Two outputs:
1. **`artists_for_verification.csv`** — one row per artist with metadata, plus blank columns for your manual judgments
2. **`songs_filtered.csv`** — the song-level corpus, automatically filtered

Open the artists CSV, fill in `L1_confirmed`, `decision_include_exclude`, etc., then back-filter songs by your verified artist list.

In [27]:
artist_summary = (
    filtered.groupby("artist_slug")
    .agg(
        artist_name=("artist_name", "first"),
        n_qualifying_songs=("song_id", "count"),
        n_with_features=("has_features", "sum"),
        mean_english_ratio=("english_ratio", "mean"),
        total_lines=("n_total_lines", "sum"),
    )
    .reset_index()
    .sort_values("n_qualifying_songs", ascending=False)
)

# Merge in candidate metadata for verification context
candidate_lookup = candidates.copy()
candidate_lookup["artist_slug"] = candidate_lookup["name"].apply(slugify)
artist_summary = artist_summary.merge(
    candidate_lookup[["artist_slug", "birthplace", "birth_year", "genre", "country", "enwiki_url"]],
    on="artist_slug", how="left"
)

artist_summary["meets_song_minimum"] = artist_summary["n_qualifying_songs"] >= CONFIG["min_songs_per_artist"]

# Blank columns for your manual verification
for col in ["L1_confirmed", "raised_in", "age_moved_if_applicable",
            "L1_confidence_high_med_low", "verification_sources",
            "decision_include_exclude", "notes"]:
    artist_summary[col] = ""

artist_summary.to_csv(DATA / "processed" / "artists_for_verification.csv", index=False)
print(f"Artists ready for manual verification: {len(artist_summary)}")
print(f"  Meeting song minimum ({CONFIG['min_songs_per_artist']}+ songs): {artist_summary['meets_song_minimum'].sum()}")
artist_summary.head(15)

Artists ready for manual verification: 11
  Meeting song minimum (1+ songs): 11


,artist_slug,artist_name,n_qualifying_songs,n_with_features,mean_english_ratio,total_lines,birthplace,birth_year,genre,country,enwiki_url,meets_song_minimum,L1_confirmed,raised_in,age_moved_if_applicable,L1_confidence_high_med_low,verification_sources,decision_include_exclude,notes
0,youssoupha,Youssoupha,2,2,0.365066,119,Kinshasa,1979.0,alternative hip-hop; hip-hop,FR,https://en.wikipedia.org/wiki/Youssoupha,True,,,,,,,
1,aloïse_sauvage,Aloïse Sauvage,1,0,0.432836,67,NaN,1992.0,hip-hop,FR,https://en.wikipedia.org/wiki/Alo%C3%AFse_Sauvage,True,,,,,,,
2,dooz_kawa,Dooz Kawa,1,1,0.187500,80,Chartres,1980.0,NaN,FR,https://en.wikipedia.org/wiki/Dooz_Kawa,True,,,,,,,
3,freeze_corleone,Freeze Corleone,1,1,0.344262,61,Les Lilas,1992.0,French hip-hop; cloud rap; drill; trap music,FR,https://en.wikipedia.org/wiki/Freeze_Corleone,True,,,,,,,
4,kaaris,Kaaris,1,1,0.500000,74,Abidjan,1980.0,French hip-hop; trap music,FR,https://en.wikipedia.org/wiki/Kaaris,True,,,,,,,
5,laylow,Laylow,1,1,0.333333,39,Toulouse,1993.0,NaN,FR,https://en.wikipedia.org/wiki/Laylow,True,,,,,,,
6,lomepal,Lomepal,1,0,0.191176,68,Paris,1991.0,French hip-hop,FR,https://en.wikipedia.org/wiki/Lomepal,True,,,,,,,
7,népal,Népal,1,1,0.217949,78,14th arrondissement of Paris,1990.0,French hip-hop,FR,https://en.wikipedia.org/wiki/N%C3%A9pal_(rapper),True,,,,,,,
8,orelsan,Orelsan,1,1,0.247863,117,Alençon,1982.0,French hip-hop,FR,https://en.wikipedia.org/wiki/Orelsan,True,,,,,,,
9,sinik,Sinik,1,0,0.523810,84,4th arrondissement of Paris,1980.0,French hip-hop,FR,https://en.wikipedia.org/wiki/Sinik,True,,,,,,,
